In [5]:
import numpy as np
import pandas as pd

In [20]:
df = pd.read_parquet("data/groups/group_1.parquet")
print(len(df))
df.head()

5058272


,Date,Expiry,Texp,z,w,DTE,Maturity-Group
index,,,,,,,
219043,2000-01-03,2000-01-22,0.052019,-0.400915,NaN,19,1m
219044,2000-01-03,2000-01-22,0.052019,-0.350905,0.000097,19,1m
219045,2000-01-03,2000-01-22,0.052019,-0.326807,0.000086,19,1m
219046,2000-01-03,2000-01-22,0.052019,-0.303277,0.000076,19,1m
219047,2000-01-03,2000-01-22,0.052019,-0.280287,0.000066,19,1m


In [21]:
def hasInsideNaN(arr):
    valArr = ~np.isnan(arr)
    
    minIndex = np.argmax(valArr)
    maxIndex = len(valArr) - 1 - np.argmax(valArr[::-1])

    return not np.all(valArr[minIndex:maxIndex + 1])

In [25]:
from tqdm import tqdm

for groupID in range(5):
    df = pd.read_parquet(f"data/groups/group_{groupID}.parquet")

    indexed =  df.groupby(["Date", "Texp"])
    kept = []
    for (date, texp), sub in tqdm(indexed):
        if hasInsideNaN(sub["w"]) or len(sub) < 20:
            continue

        kept.append(sub)

    dfCompact = pd.concat(kept, ignore_index=True)
    dfCompact.to_parquet(f"data/groups/compact_group_{groupID}.parquet", index=False)
        

100%|██████████| 27025/27025 [00:06<00:00, 4206.54it/s]


In [60]:
groupID = 0
df = pd.read_parquet(f"data/groups/compact_group_{groupID}.parquet")


X = []
X_keys = []
zgrid = np.linspace(-1, 1, 50)
for key, sub in tqdm(df.groupby(["Date", "Texp"])):
    sub = sub.dropna(subset=["z", "w"])
    z = sub["z"].to_numpy()
    w = sub["w"].to_numpy()
    
    z_max_abs = np.abs(z).max()
    
    if z_max_abs == 0:
        continue
    
    z_norm = z / z_max_abs
    w_norm = w / w.mean()
    
    w_interp = np.interp(zgrid, z_norm, w_norm, left=np.nan, right=np.nan)
    X.append(w_interp)
    X_keys.append(key)

100%|██████████| 4170/4170 [00:03<00:00, 1113.69it/s]


In [62]:
from scipy.spatial.distance import pdist, squareform

def dist(a, b):
    a = np.array(a)
    b = np.array(b)

    mask = ~np.isnan(a) & ~np.isnan(b)

    if mask.sum() == 0:
        return 100

    diff = a[mask] - b[mask]
    return np.sqrt(np.sum(diff**2) / mask.sum())
    

X = np.array(X[:1000])
distances = pdist(X, metric=lambda u, v: dist(u, v))
np.save(f"dist_group_{groupID}.npy", distances)

In [63]:
D = squareform(distances)

In [64]:
from sklearn.cluster import DBSCAN
model = DBSCAN(metric="precomputed", eps=0.5, min_samples=5)
labels = model.fit_predict(D)

In [66]:
keys = np.array(X_keys[:1000])
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

for groupID in [0, 1, 2, 3, 4]:
    if groupID != 0:
        continue

    # ---- attach cluster labels per curve ----
    curve_df = pd.DataFrame(keys, columns=["Date", "Texp"])
    curve_df["CL"] = labels

    df = df.merge(curve_df, on=["Date", "Texp"], how="left")
    print(df)
    n_clusters = len(np.unique(labels))

    # global z-grid
    common_z = zgrid

    for cluster in np.unique(labels):
        plt.figure()

        cluster_df = df[df["CL"] == cluster]
        interpolated_curves = []

        # ---- interpolate each curve ----
        for (date, texp), sub in cluster_df.groupby(["Date", "Texp"]):

            sub = sub.dropna(subset=["z", "w"])
            sub = sub.sort_values("z")

            if len(sub) < 20:
                continue

            f = interp1d(
                sub["z"],
                sub["w"],
                kind="linear",
                bounds_error=False,
                fill_value=np.nan
            )

            w_interp = f(common_z)
            interpolated_curves.append(w_interp)

            plt.plot(sub["z"], sub["w"], color="gray", alpha=0.2)

        interpolated_curves = np.array(interpolated_curves)

        print("cluster", cluster, "curves:", interpolated_curves.shape)

        mean_w = np.nanmean(interpolated_curves, axis=0)

        plt.plot(
            common_z,
            mean_w,
            color="red",
            linewidth=3,
            label="Mean curve"
        )

        plt.xlabel("z")
        plt.ylabel("w")
        plt.title(f"Cluster {cluster}")
        plt.legend()
        # plt.show()
        # plt.savefig(f"images/clusters/hierarchy/group_{groupID}_cluster_{cluster}.png")

             Date     Expiry      Texp         z   w  DTE Maturity-Group  \
0      2000-01-20 2000-01-22  0.005476 -0.394442 NaN    2          <=1wk   
1      2000-01-20 2000-01-22  0.005476 -0.344432 NaN    2          <=1wk   
2      2000-01-20 2000-01-22  0.005476 -0.320334 NaN    2          <=1wk   
3      2000-01-20 2000-01-22  0.005476 -0.296804 NaN    2          <=1wk   
4      2000-01-20 2000-01-22  0.005476 -0.273814 NaN    2          <=1wk   
...           ...        ...       ...       ...  ..  ...            ...   
648833 2024-12-30 2025-01-06     0.016  0.169500 NaN    7          <=1wk   
648834 2024-12-30 2025-01-06     0.016  0.197671 NaN    7          <=1wk   
648835 2024-12-30 2025-01-06     0.016  0.225070 NaN    7          <=1wk   
648836 2024-12-30 2025-01-06     0.016  0.251738 NaN    7          <=1wk   
648837 2024-12-30 2025-01-06     0.016  0.277714 NaN    7          <=1wk   

        CL_x  CL_y  
0        0.0   0.0  
1        0.0   0.0  
2        0.0   0.0  
3  

KeyError: 'CL'

<Figure size 640x480 with 0 Axes>